# Data Cleaning

## Import thư viện

In [55]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd

In [56]:
PROJECT_ROOT = Path("..").resolve()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
TABLE_DIR = OUTPUT_DIR / "tables"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

raw_train_path = RAW_DIR / "raw_data_train.csv"
raw_test_path = RAW_DIR / "raw_data_test.csv"

print("Raw train path:", raw_train_path)
print("Raw test path:", raw_test_path)
print("Clean dir:", CLEAN_DIR)

Raw train path: D:\DataScientFinalProject\data\raw\raw_data_train.csv
Raw test path: D:\DataScientFinalProject\data\raw\raw_data_test.csv
Clean dir: D:\DataScientFinalProject\data\clean


## Load raw data đã lưu ra csv

In [57]:
raw_train = pd.read_csv(raw_train_path)
raw_test = pd.read_csv(raw_test_path)

print("Raw train shape:", raw_train.shape)
print("Raw test shape:", raw_test.shape)

display(raw_train.head())
display(raw_test.head())

print("Train columns:")
print(raw_train.columns.tolist())

print("\nTest columns:")
print(raw_test.columns.tolist())

print("\nSame columns:", list(raw_train.columns) == list(raw_test.columns))

Raw train shape: (546190, 14)
Raw test shape: (60688, 14)


,id,job_title,company_name,salary,location,job_type,job_industry,experience_level,education_level,job_position,job_description,benefits,requirements,year
0,301834,Nhân Viên Kinh Doanh - Thu Nhập Đến 30 Triệu (...,Công Ty TNHH Bê Tông Trang Trí Việt Nam,12.000.000 - 30.000.000 VND,"Tầng 5, Tòa nhà Phú Hưng 298 Ung Văn Khiêm | 5...",Toàn thời gian,Xây dựng,6 năm,Cao đẳng,Nhân viên,"Lên kế hoạch tiếp cận, chăm sóc khách hàng là ...",Thu nhập: Từ 12- 30 triệu ( Lương cứng + hoa h...,"Độ tuổi từ 27-40, tốt nghiệp cao đẳng, đại học...",2024
1,477331,NV Pha Chế. Caffe,Caffe,6.000.000 - 10.000.000 VND,880 tỉnh lộ 43 phường bình chiểu,Remote,Bán hàng - Kinh doanh,2 năm,Không,Nhân viên,"Tìm kiếm khách hàng tiềm năng, mở rộng nguồn k...",Lương cơ bản + hoa hồng + thưởng KPI - Được hư...,"Nam/Nữ Từ 22 – 35 tuổi, có đam mê và nhiệt huy...",2025
2,540064,Kỹ Sư Giám Sát Xây Dựng,Công Ty TNHH XD TM DV Không Gian Đẹp,10.000.000 - 15.000.000 VND,"68 Song Hành, Quốc lộ 22, Trung Chánh | 72/3C ...",Toàn thời gian,Xây dựng,5 năm,Đại học,Nhân viên,"• Giám sát, Quản lý các tổ đội, từ phần thô đế...",• Mức lương: 10 - 14tr + Phụ cấp công việc • P...,• Tốt nghiệp Đại học chuyên ngành Xây dựng Dân...,2025
3,114270,"Tuyển Sales, Telesales Cho Công Ty Bhnt Pruden...",Công Ty TNHH Một Thành Viên Đại Lý Bảo Hiểm Gl,5.000.000 - 7.000.000 VND,"153 Cách Mạng Tháng Tám, Phường Hoa Lư, Thành ...",Toàn thời gian,Chưa xác định,1 năm,Trung học,Chưa cập nhật,"Gọi điện thoại, kết nối và lên hẹn với khách h...",Được hưởng đầy đủ chế độ BHXH và nghỉ lễ của n...,"cần ứng viên từ 21 tuổi, tốt nghiệp THPT trở lên",2022
4,322289,Kế Toán Tổng Hợp,CÔNG TY TNHH XUẤT NHẬP KHẨU Ô TÔ MIỀN NAM,8.000.000 - 10.000.000 VND,"159 Nguyễn Chí Thanh | 68 QL1A, Phường An Phú ...",Toàn thời gian,Kế toán / Kiểm toán,5 năm,Cao đẳng,Nhân viên,− Thực hiện hiệu quả sổ sách kế toán các nghiệ...,− Lương cơ bản: 8.000.000 đồng – 10.000.000 đồ...,"− Trình độ: Tốt nghiệp Cao đẳng, Đại học chuyê...",2024


,id,job_title,company_name,salary,location,job_type,job_industry,experience_level,education_level,job_position,job_description,benefits,requirements,year
0,317083,Nhân Viên Đối Ngoại,Công Ty TNHH Đầu Tư Và Phát Triển Nguồn Nhân L...,8.000.000 - 11.000.000 VND,"TT20 Trịnh Văn Bô | Số 17A, Tổ dân phố số 5 - ...",Toàn thời gian,Giáo dục - Đào tạo / Biên phiên dịch,2 năm,Đại học,Nhân viên,Phụ trách công việc: trực chát với đối tác Đài...,Lương: cơ bản (theo thoả thuận) + doanh số xuấ...,"Tốt nghiệp đại học, có chứng chỉ tiếng trung H...",2024
1,384327,Giáo Viên Toeic Full / Part-Time,Công Ty TNHH Một Thành Viên Đoàn Phan Gia Lâm,5.000.000 - 15.000.000 VND,"30 Trần Quang Diệu | 31 Trương Văn Đa, Hòa Khá...",Bán thời gian,Giáo dục - Đào tạo / Chăm sóc khách hàng,1 năm,Không,Nhân viên,Thực hiện công việc giảng dạy theo đúng mục ti...,Lương cơ bản từ 5 triệu cho part-time và 8-15 ...,"Có chứng chỉ TOEIC 850+ hoặc IELTS từ 7.5, phá...",2024
2,509652,Nhân Viên Mua Hàng - Xử Lý Đơn Hàng,Công Ty TNHH Nhựa Song Mộc,10.000.000 - 12.000.000 VND,"Đường số 11, Khu công nghiệp Tân Đức, Xã Hựu T...",Toàn thời gian,Kế toán / Kiểm toán,3 năm,Trung cấp,Nhân viên,"Nhân viên mua hàng, nhận và xử lý đơn hàng, xu...",Tổng thu nhập từ 10 triệu đến 12 triệu đồng ( ...,Độ tuổi từ 22-35t - Trình độ: Tốt nghiệp trung...,2025
3,278737,Nhân Viên Quản Lý Dự Án Chuẩn Bị Sản Xuất Sản ...,Công Ty TNHH Yamaha Motor Việt Nam,10.000.000 - 13.000.000 VND,Nhà máy Yamaha Motor Việt Nam – Khu CN Nội Bài...,Toàn thời gian,Cơ khí - Ô tô - Tự động hóa / Sản xuất - Lắp r...,3 năm,Đại học,Nhân viên,Đề xuất mục tiêu và chuẩn bị kế hoạch thực hiệ...,Địa điểm làm việc: Nhà máy Yamaha Motor Việt N...,"Tốt nghiệp Đại học; - Tiếng Anh tốt, thành thạ...",2023
4,468353,Kỹ Thuật Viên (Không Yêu Cầu Kinh Nghiệm),VPĐD AMAZON PAPYRUS CHEMICALS (VIETNAM) LIMITE...,8.000.000 - 10.000.000 VND,"Cụm CN Phú Lâm/ Phong Khê | Tầng 7, Tòa nhà To...",Toàn thời gian,Chưa xác định,1 năm,Cao đẳng,Nhân viên,"Dịch vụ kỹ thuật, theo dõi hoá chất sử dụng tạ...",Mức lương thỏa thuận theo năng lực - Các chế đ...,"Tốt nghiệp Cao Ðẳng, Ðại học chuyên ngành Công...",2025


Train columns:
['id', 'job_title', 'company_name', 'salary', 'location', 'job_type', 'job_industry', 'experience_level', 'education_level', 'job_position', 'job_description', 'benefits', 'requirements', 'year']

Test columns:
['id', 'job_title', 'company_name', 'salary', 'location', 'job_type', 'job_industry', 'experience_level', 'education_level', 'job_position', 'job_description', 'benefits', 'requirements', 'year']

Same columns: True


## Kiểm tra duplicate data

In [58]:
train_duplicate_all = raw_train.duplicated().sum()
test_duplicate_all = raw_test.duplicated().sum()

print("Duplicate rows in raw_train:", train_duplicate_all)
print("Duplicate rows in raw_test:", test_duplicate_all)

# Kiểm tra duplicate data dựa trên cột "id"
if "id" in raw_train.columns:
    print("Duplicated id in train:", raw_train["id"].duplicated().sum())
    print("Duplicated id in test:", raw_test["id"].duplicated().sum())

Duplicate rows in raw_train: 0
Duplicate rows in raw_test: 0
Duplicated id in train: 0
Duplicated id in test: 0


## Hàm làm sạch text

### Chuẩn hóa Unicode và text

In [59]:
def normalize_unicode(text):
    """
        Chuẩn hóa Unicode tiếng việt về dạng NFC.
        Giúp các ký tự tiếng Việt có dấu được biểu diễn nhất quán.
    """
    if pd.isna(text):
        return ""
    return unicodedata.normalize("NFC", text)

def clean_basic_text(text):
    """
        Làm sạch text cơ bản:
        - chuyển NaN thành chuỗi rỗng
        - chuẩn hóa Unicode
        - chuyển về lowercase
        - xóa một số ký tự điều khiển
        - chuẩn hóa khoảng trắng
    """
    text = normalize_unicode(text)
    text = text.lower()
    
    # Thay kí tự xuống dòng/tab bằng khoảng trắng
    text = re.sub(r"[\r\n\t]+", " ", text)
    
    # xóa ký tự điều khiển lạ
    text = re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", text)
    
    # chuẩn hóa khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

***Giải thích***: Hàm `clean_basic_text` không xóa hết dấu câu hay tiếng Anh. Vì trong tuyển dụng, các từ như `python`, `java`, `sql`, `sales`, `marketing`, `backend` rất quan trọng. Ta chỉ làm sạch cơ bản để text đồng nhất hơn, còn các từ khóa vẫn được giữ lại để phục vụ cho việc phân tích sau này.

## Làm sạch text columns

In [60]:
TEXT_COLUMNS = ["job_title", "job_description", "requirements", "benefits"]

def clean_text_columns(df):
    df = df.copy()
    
    for col in TEXT_COLUMNS:
        if col in df.columns:
            df[col] = df[col].fillna("").apply(clean_basic_text)
            
    return df

clean_train = clean_text_columns(raw_train)
clean_test = clean_text_columns(raw_test)

display(clean_train[TEXT_COLUMNS].head())

,job_title,job_description,requirements,benefits
0,nhân viên kinh doanh - thu nhập đến 30 triệu (...,"lên kế hoạch tiếp cận, chăm sóc khách hàng là ...","độ tuổi từ 27-40, tốt nghiệp cao đẳng, đại học...",thu nhập: từ 12- 30 triệu ( lương cứng + hoa h...
1,nv pha chế. caffe,"tìm kiếm khách hàng tiềm năng, mở rộng nguồn k...","nam/nữ từ 22 – 35 tuổi, có đam mê và nhiệt huy...",lương cơ bản + hoa hồng + thưởng kpi - được hư...
2,kỹ sư giám sát xây dựng,"• giám sát, quản lý các tổ đội, từ phần thô đế...",• tốt nghiệp đại học chuyên ngành xây dựng dân...,• mức lương: 10 - 14tr + phụ cấp công việc • p...
3,"tuyển sales, telesales cho công ty bhnt pruden...","gọi điện thoại, kết nối và lên hẹn với khách h...","cần ứng viên từ 21 tuổi, tốt nghiệp thpt trở lên",được hưởng đầy đủ chế độ bhxh và nghỉ lễ của n...
4,kế toán tổng hợp,− thực hiện hiệu quả sổ sách kế toán các nghiệ...,"− trình độ: tốt nghiệp cao đẳng, đại học chuyê...",− lương cơ bản: 8.000.000 đồng – 10.000.000 đồ...


## Làm sạch categorical columns 

In [61]:
CATEGORICAL_COLUMNS = [
    "company_name",
    "location",
    "job_type",
    "job_industry",
    "experience_level",
    "education_level",
    "job_position",
]

def clean_categorical_value(value):
    """
        Làm sạch giá trị categorical:
        - NaN -> "unknown"
        - chuẩn hóa Unicode
        - Xóa khoảng trắng thừa
    """
    if pd.isna(value):
        return "Unknown"
    
    value = unicodedata.normalize("NFC", str(value))
    value = re.sub(r"[\r\n\t]+", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    
    if value == "":
        return "Unknown"
    
    return value

def clean_categorical_columns(df):
    df = df.copy()
    
    for col in CATEGORICAL_COLUMNS:
        if col in df.columns:
            df[col] = df[col].apply(clean_categorical_value)
            
    return df

clean_train = clean_categorical_columns(clean_train)
clean_test = clean_categorical_columns(clean_test)

## Parse salary 
Dữ liệu lương có thể có nhiều dạng như:
- 12.000.000 - 15.000.000 VND
- 6.000.000 - 10.000.000 VND
- Thỏa thuận
- Cạnh tranh
- Tới 30 triệu
- 1000 - 1500 USD

Nên ta sẽ chuẩn hóa lương về dạng: triệu VND/tháng. 
Nếu là USD, tạm thời không quy đổi. Nên đánh dấu `salary_currency` = `USD`

In [62]:
def parse_number_token(token):
    """
    Parse một token số trong salary.

    Hàm này xử lý các dạng phổ biến:
    - 12.000.000  -> 12000000
    - 500.000     -> 500000
    - 12,000,000  -> 12000000
    - 12.5        -> 12.5
    - 12,5        -> 12.5

    Lý do cần hàm này:
    Nếu parse đơn giản bằng float, chuỗi '500.000 VND' có thể bị hiểu thành 500.0.
    Trong ngữ cảnh Việt Nam, '500.000 VND' thường là 500 nghìn VND, tức 0.5 triệu VND.
    """
    token = str(token).strip()

    # Dạng phân cách nghìn bằng dấu chấm: 12.000.000 hoặc 500.000
    if "." in token:
        parts = token.split(".")
        if len(parts) > 1 and all(len(part) == 3 for part in parts[1:]):
            return float(token.replace(".", ""))

    # Dạng phân cách nghìn bằng dấu phẩy: 12,000,000 hoặc 500,000
    if "," in token:
        parts = token.split(",")
        if len(parts) > 1 and all(len(part) == 3 for part in parts[1:]):
            return float(token.replace(",", ""))

    # Trường hợp còn lại: xem dấu phẩy là dấu thập phân
    token = token.replace(",", ".")
    return float(token)


def parse_salary_to_million_vnd(salary):
    """
    Parse salary từ chuỗi gốc sang các biến có cấu trúc:

    - salary_available
    - salary_min_million_vnd
    - salary_max_million_vnd
    - salary_avg_million_vnd
    - salary_currency

    Quy ước:
    - Với VND, chuẩn hóa về đơn vị triệu VND/tháng.
    - Với USD, chỉ đánh dấu currency = USD, chưa quy đổi sang VND.
    - Với salary dạng thỏa thuận/cạnh tranh/đang cập nhật, đặt salary_available = 0.
    """
    if pd.isna(salary):
        return pd.Series({
            "salary_available": 0,
            "salary_min_million_vnd": np.nan,
            "salary_max_million_vnd": np.nan,
            "salary_avg_million_vnd": np.nan,
            "salary_currency": "Unknown"
        })

    text = str(salary).strip().lower()

    unavailable_keywords = [
        "thỏa thuận",
        "thoả thuận",
        "cạnh tranh",
        "đang cập nhật",
        "không công khai",
        "unknown"
    ]

    if text == "" or any(keyword in text for keyword in unavailable_keywords):
        return pd.Series({
            "salary_available": 0,
            "salary_min_million_vnd": np.nan,
            "salary_max_million_vnd": np.nan,
            "salary_avg_million_vnd": np.nan,
            "salary_currency": "Unknown"
        })

    currency = "VND"
    if "usd" in text or "$" in text:
        currency = "USD"

    # Lấy tất cả cụm số
    numbers = re.findall(r"\d+(?:[.,]\d+)*", text)

    if len(numbers) == 0:
        return pd.Series({
            "salary_available": 0,
            "salary_min_million_vnd": np.nan,
            "salary_max_million_vnd": np.nan,
            "salary_avg_million_vnd": np.nan,
            "salary_currency": "Unknown"
        })

    parsed_numbers = []
    for token in numbers:
        try:
            value = parse_number_token(token)
            parsed_numbers.append(value)
        except:
            pass

    if len(parsed_numbers) == 0:
        return pd.Series({
            "salary_available": 0,
            "salary_min_million_vnd": np.nan,
            "salary_max_million_vnd": np.nan,
            "salary_avg_million_vnd": np.nan,
            "salary_currency": "Unknown"
        })

    # Nếu là USD thì không quy đổi ở giai đoạn này
    if currency == "USD":
        return pd.Series({
            "salary_available": 1,
            "salary_min_million_vnd": np.nan,
            "salary_max_million_vnd": np.nan,
            "salary_avg_million_vnd": np.nan,
            "salary_currency": "USD"
        })

    # Nếu là VND, chuẩn hóa về triệu VND
    converted = []

    for value in parsed_numbers:
        # Nếu số >= 1,000,000 thì đang là đơn vị VND
        # Ví dụ 12.000.000 -> 12 triệu
        if value >= 1_000_000:
            converted.append(value / 1_000_000)

        # Nếu text có đơn vị triệu/trieu hoặc dạng viết tắt tr, xem số là triệu
        # Ví dụ 15 triệu, 15 tr -> 15 triệu
        elif any(unit in text for unit in ["triệu", "trieu", " tr", "tr "]):
            converted.append(value)

        # Nếu có VND/VNĐ/đ mà số dưới 1,000,000
        # Ví dụ 500.000 VND -> 0.5 triệu
        elif any(unit in text for unit in ["vnd", "vnđ", "đ"]):
            converted.append(value / 1_000_000)

        # Fallback: nếu không rõ đơn vị, giữ là triệu
        else:
            converted.append(value)

    if len(converted) == 0:
        return pd.Series({
            "salary_available": 0,
            "salary_min_million_vnd": np.nan,
            "salary_max_million_vnd": np.nan,
            "salary_avg_million_vnd": np.nan,
            "salary_currency": "Unknown"
        })

    salary_min = min(converted)
    salary_max = max(converted)
    salary_avg = (salary_min + salary_max) / 2

    return pd.Series({
        "salary_available": 1,
        "salary_min_million_vnd": salary_min,
        "salary_max_million_vnd": salary_max,
        "salary_avg_million_vnd": salary_avg,
        "salary_currency": "VND"
    })


def add_salary_features(df):
    """
    Thêm các feature salary vào dataframe.
    Đồng thời fill salary gốc bị thiếu thành 'Unknown'.
    """
    df = df.copy()

    if "salary" in df.columns:
        df["salary"] = df["salary"].fillna("Unknown")
        salary_features = df["salary"].apply(parse_salary_to_million_vnd)
        df = pd.concat([df, salary_features], axis=1)

        salary_text = df["salary"].astype(str).str.strip().str.lower()

        unavailable_pattern = (
            r"thỏa thuận|thoả thuận|thương lượng|tuỳ năng lực|tùy năng lực|"
            r"cạnh tranh|đang cập nhật|không công khai|vnd month|you'll love it|unknown"
        )

        is_unavailable = (
            (salary_text == "")
            | salary_text.str.contains(unavailable_pattern, regex=True, na=False)
        )

        has_number = salary_text.str.contains(r"\d", regex=True, na=False)

        df["salary_parse_issue"] = (
            (~is_unavailable)
            & has_number
            & (
                (df["salary_currency"] == "VND")
                & (
                    df["salary_avg_million_vnd"].isna()
                    | (df["salary_min_million_vnd"] <= 0)
                    | (df["salary_max_million_vnd"] <= 0)
                    | (df["salary_min_million_vnd"] > df["salary_max_million_vnd"])
                    | (df["salary_avg_million_vnd"] > 200)
                )
            )
        ).astype(int)

    return df


clean_train = add_salary_features(clean_train)
clean_test = add_salary_features(clean_test)

display(clean_train[
    [
        "salary",
        "salary_available",
        "salary_min_million_vnd",
        "salary_max_million_vnd",
        "salary_avg_million_vnd",
        "salary_currency",
        "salary_parse_issue"
    ]
].head(20))

,salary,salary_available,salary_min_million_vnd,salary_max_million_vnd,salary_avg_million_vnd,salary_currency,salary_parse_issue
0,12.000.000 - 30.000.000 VND,1,12.0,30.0,21.00,VND,0
1,6.000.000 - 10.000.000 VND,1,6.0,10.0,8.00,VND,0
2,10.000.000 - 15.000.000 VND,1,10.0,15.0,12.50,VND,0
3,5.000.000 - 7.000.000 VND,1,5.0,7.0,6.00,VND,0
4,8.000.000 - 10.000.000 VND,1,8.0,10.0,9.00,VND,0
5,8.000.000 - 10.000.000 VND,1,8.0,10.0,9.00,VND,0
6,7.000.000 - 16.000.000 VND,1,7.0,16.0,11.50,VND,0
7,7.000.000 - 10.000.000 VND,1,7.0,10.0,8.50,VND,0
8,6000000 - 10000000 VND MONTH,1,6.0,10.0,8.00,VND,0
9,6.000.000 - 15.000.000 VND,1,6.0,15.0,10.50,VND,0


In [63]:
salary_test_cases = pd.Series([
    "15 - 20 triệu",
    "12.000.000 - 18.000.000 VND",
    "500.000 VND",
    "1.000.000 - 2.000.000 VND",
    "Thỏa thuận",
    "Cạnh tranh",
    "Đang cập nhật",
    "1000 - 1500 USD",
    "Tới 30 triệu",
    "Trên 15 triệu",
    "VND MONTH",
    None,
])

salary_test_result = salary_test_cases.apply(parse_salary_to_million_vnd)

salary_test_df = pd.concat(
    [
        salary_test_cases.rename("salary_raw"),
        salary_test_result
    ],
    axis=1
)

display(salary_test_df)

,salary_raw,salary_available,salary_min_million_vnd,salary_max_million_vnd,salary_avg_million_vnd,salary_currency
0,15 - 20 triệu,1,15.0,20.0,17.5,VND
1,12.000.000 - 18.000.000 VND,1,12.0,18.0,15.0,VND
2,500.000 VND,1,0.5,0.5,0.5,VND
3,1.000.000 - 2.000.000 VND,1,1.0,2.0,1.5,VND
4,Thỏa thuận,0,NaN,NaN,NaN,Unknown
5,Cạnh tranh,0,NaN,NaN,NaN,Unknown
6,Đang cập nhật,0,NaN,NaN,NaN,Unknown
7,1000 - 1500 USD,1,NaN,NaN,NaN,USD
8,Tới 30 triệu,1,30.0,30.0,30.0,VND
9,Trên 15 triệu,1,15.0,15.0,15.0,VND


In [64]:
salary_anomalies = clean_train[
    clean_train["salary_parse_issue"] == 1
][
    [
        "salary",
        "salary_available",
        "salary_min_million_vnd",
        "salary_max_million_vnd",
        "salary_avg_million_vnd",
        "salary_currency",
        "salary_parse_issue"
    ]
]

print("Number of salary anomalies:", len(salary_anomalies))
display(salary_anomalies.head(50))

salary_anomalies.to_csv(
    TABLE_DIR / "stage_03_salary_anomalies.csv",
    index=False,
    encoding="utf-8-sig"
)

Number of salary anomalies: 7688


,salary,salary_available,salary_min_million_vnd,salary_max_million_vnd,salary_avg_million_vnd,salary_currency,salary_parse_issue
409,0 VND,1,0.0,0.0,0.0,VND,1
601,0 VND,1,0.0,0.0,0.0,VND,1
607,0 VND,1,0.0,0.0,0.0,VND,1
789,0 VND,1,0.0,0.0,0.0,VND,1
797,0 VND,1,0.0,0.0,0.0,VND,1
827,0 VND,1,0.0,0.0,0.0,VND,1
908,0 VND,1,0.0,0.0,0.0,VND,1
946,0 VND,1,0.0,0.0,0.0,VND,1
974,0 VND,1,0.0,0.0,0.0,VND,1
988,0 VND,1,0.0,0.0,0.0,VND,1


## Xử lý year - Làm sạch dữ liệu năm 

In [65]:
def clean_year_column(df):
    df = df.copy()
    
    if "year" in df.columns:
        df["year"] = pd.to_numeric(df["year"], errors="coerce")
        
    return df

In [66]:
clean_train = clean_year_column(clean_train)
clean_test = clean_year_column(clean_test)

print(clean_train["year"].value_counts(dropna=False).sort_index())

year
2022    100053
2023    116761
2024    142985
2025    144378
2026     42013
Name: count, dtype: int64


## Tạo các feature text length

In [67]:
def add_text_length_features(df):
    df = df.copy()
    
    for col in TEXT_COLUMNS:
        if col in df.columns:
            df[f"{col}_char_len"] = df[col].str.len()
            df[f"{col}_word_count"] = df[col].str.split().apply(len)
            
    return df

In [68]:
clean_train = add_text_length_features(clean_train)
clean_test = add_text_length_features(clean_test)

display(clean_train[[
    "job_title_char_len",
    "job_title_word_count",
    "job_description_char_len",
    "job_description_word_count",
    "requirements_char_len",
    "requirements_word_count",
    "benefits_char_len",
    "benefits_word_count",
]].head())

,job_title_char_len,job_title_word_count,job_description_char_len,job_description_word_count,requirements_char_len,requirements_word_count,benefits_char_len,benefits_word_count
0,58,13,480,108,607,136,416,92
1,17,4,610,136,200,45,205,45
2,23,6,475,108,343,78,339,74
3,50,8,157,34,48,11,158,35
4,16,4,637,144,323,71,513,105


## Kiểm tra sau khi cleaning

In [69]:
# Missing value sau khi đã clean
clean_missing_summary = pd.DataFrame({
    "missing_count": clean_train.isna().sum(),
    "missing_ratio": clean_train.isna().mean()
}).sort_values("missing_ratio", ascending=False)

display(clean_missing_summary)

# Kiểm tra shape
print("Raw train shape:", raw_train.shape)
print("Clean train shape:", clean_train.shape)

print("Raw test shape:", raw_test.shape)
print("Clean test shape:", clean_test.shape)

# Kiểm tra trùng train/test sau khi clean
if "id" in clean_train.columns and "id" in clean_test.columns:
    overlap_ids_after_clean = set(clean_train["id"]).intersection(set(clean_test["id"]))
    print("Overlapped ids after cleaning:", len(overlap_ids_after_clean))

,missing_count,missing_ratio
salary_max_million_vnd,10041,0.018384
salary_avg_million_vnd,10041,0.018384
salary_min_million_vnd,10041,0.018384
id,0,0.000000
location,0,0.000000
job_type,0,0.000000
job_industry,0,0.000000
experience_level,0,0.000000
education_level,0,0.000000
job_title,0,0.000000


Raw train shape: (546190, 14)
Clean train shape: (546190, 28)
Raw test shape: (60688, 14)
Clean test shape: (60688, 28)
Overlapped ids after cleaning: 0


## Lưu clean data

In [70]:
clean_train_path = CLEAN_DIR / "clean_data_train.csv"
clean_test_path = CLEAN_DIR / "clean_data_test.csv"

clean_train.to_csv(clean_train_path, index=False, encoding="utf-8-sig")
clean_test.to_csv(clean_test_path, index=False, encoding="utf-8-sig")

print("Saved clean train:", clean_train_path)
print("Saved clean test:", clean_test_path)

Saved clean train: D:\DataScientFinalProject\data\clean\clean_data_train.csv
Saved clean test: D:\DataScientFinalProject\data\clean\clean_data_test.csv


In [71]:
clean_train.to_csv(PROJECT_ROOT / "clean_data_train.csv", index=False, encoding="utf-8-sig")
clean_test.to_csv(PROJECT_ROOT / "clean_data_test.csv", index=False, encoding="utf-8-sig")

print("Copied clean data to project root.")

Copied clean data to project root.


## Lưu bảng thống kê Giai đoạn 3 - Metadata cleaning

In [72]:
stage_03_metadata = {
    "n_raw_train": len(raw_train),
    "n_clean_train": len(clean_train),
    "n_raw_test": len(raw_test),
    "n_clean_test": len(clean_test),

    "n_columns_raw_train": raw_train.shape[1],
    "n_columns_clean_train": clean_train.shape[1],

    "overlap_ids_after_clean": len(overlap_ids_after_clean) if "overlap_ids_after_clean" in globals() else None,

    "salary_available_train_ratio": clean_train["salary_available"].mean() if "salary_available" in clean_train.columns else None,
    "salary_available_test_ratio": clean_test["salary_available"].mean() if "salary_available" in clean_test.columns else None,

    "salary_parse_issue_train_count": int(clean_train["salary_parse_issue"].sum()) if "salary_parse_issue" in clean_train.columns else None,
    "salary_parse_issue_test_count": int(clean_test["salary_parse_issue"].sum()) if "salary_parse_issue" in clean_test.columns else None,

    "salary_parse_issue_train_ratio": clean_train["salary_parse_issue"].mean() if "salary_parse_issue" in clean_train.columns else None,
    "salary_parse_issue_test_ratio": clean_test["salary_parse_issue"].mean() if "salary_parse_issue" in clean_test.columns else None,
}

stage_03_metadata_df = pd.DataFrame([stage_03_metadata])
stage_03_metadata_df.to_csv(
    TABLE_DIR / "stage_03_metadata.csv",
    index=False,
    encoding="utf-8-sig"
)

display(stage_03_metadata_df)

,n_raw_train,n_clean_train,n_raw_test,n_clean_test,n_columns_raw_train,n_columns_clean_train,overlap_ids_after_clean,salary_available_train_ratio,salary_available_test_ratio,salary_parse_issue_train_count,salary_parse_issue_test_count,salary_parse_issue_train_ratio,salary_parse_issue_test_ratio
0,546190,546190,60688,60688,14,28,0,0.981929,0.981858,7688,839,0.014076,0.013825


## Lưu missing summary sau cleaning

In [73]:
clean_missing_summary = clean_missing_summary.reset_index().rename(columns={"index": "column"})

clean_missing_summary.to_csv(
    TABLE_DIR / "stage_03_missing_summary_clean_train.csv",
    index=False,
    encoding="utf-8-sig"
)

display(clean_missing_summary)

,column,missing_count,missing_ratio
0,salary_max_million_vnd,10041,0.018384
1,salary_avg_million_vnd,10041,0.018384
2,salary_min_million_vnd,10041,0.018384
3,id,0,0.000000
4,location,0,0.000000
5,job_type,0,0.000000
6,job_industry,0,0.000000
7,experience_level,0,0.000000
8,education_level,0,0.000000
9,job_title,0,0.000000


In [74]:
salary_parse_summary = {}

salary_parse_summary["salary_available_counts"] = (
    clean_train["salary_available"]
    .value_counts(dropna=False)
    .rename_axis("salary_available")
    .reset_index(name="count")
)

salary_parse_summary["salary_currency_counts"] = (
    clean_train["salary_currency"]
    .value_counts(dropna=False)
    .rename_axis("salary_currency")
    .reset_index(name="count")
)

salary_numeric_cols = [
    "salary_min_million_vnd",
    "salary_max_million_vnd",
    "salary_avg_million_vnd"
]

salary_numeric_describe_all = (
    clean_train[salary_numeric_cols]
    .describe()
    .T
    .reset_index()
    .rename(columns={"index": "column"})
)

salary_numeric_describe_no_issue = (
    clean_train[clean_train["salary_parse_issue"] == 0][salary_numeric_cols]
    .describe()
    .T
    .reset_index()
    .rename(columns={"index": "column"})
)

salary_parse_summary["salary_available_counts"].to_csv(
    TABLE_DIR / "stage_03_salary_available_counts.csv",
    index=False,
    encoding="utf-8-sig"
)

salary_parse_summary["salary_currency_counts"].to_csv(
    TABLE_DIR / "stage_03_salary_currency_counts.csv",
    index=False,
    encoding="utf-8-sig"
)

salary_numeric_describe_all.to_csv(
    TABLE_DIR / "stage_03_salary_numeric_describe_all.csv",
    index=False,
    encoding="utf-8-sig"
)

salary_numeric_describe_no_issue.to_csv(
    TABLE_DIR / "stage_03_salary_numeric_describe_no_issue.csv",
    index=False,
    encoding="utf-8-sig"
)

display(salary_parse_summary["salary_available_counts"])
display(salary_parse_summary["salary_currency_counts"])

print("Salary numeric describe - all:")
display(salary_numeric_describe_all)

print("Salary numeric describe - no issue:")
display(salary_numeric_describe_no_issue)

,salary_available,count
0,1,536320
1,0,9870


,salary_currency,count
0,VND,536149
1,Unknown,9870
2,USD,171


Salary numeric describe - all:


,column,count,mean,std,min,25%,50%,75%,max
0,salary_min_million_vnd,536149.0,10.474393,25.438571,0.0,7.0,9.0,11.0,1500.0
1,salary_max_million_vnd,536149.0,19.132327,85.936355,0.0,10.0,15.0,20.0,5000.0
2,salary_avg_million_vnd,536149.0,14.803360,54.802825,0.0,9.0,12.0,15.0,3000.0


Salary numeric describe - no issue:


,column,count,mean,std,min,25%,50%,75%,max
0,salary_min_million_vnd,528461.0,9.934932,5.002893,0.0,7.0,9.0,12.0,200.0
1,salary_max_million_vnd,528461.0,17.116656,12.299073,0.0,10.0,15.0,20.0,303.0
2,salary_avg_million_vnd,528461.0,13.525794,7.891160,0.0,9.0,12.0,15.0,200.0


In [75]:
salary_check_sample = clean_train[
    [
        "salary",
        "salary_available",
        "salary_min_million_vnd",
        "salary_max_million_vnd",
        "salary_avg_million_vnd",
        "salary_currency",
        "salary_parse_issue"
    ]
].sample(100, random_state=42)

salary_check_sample.to_csv(
    TABLE_DIR / "stage_03_salary_parse_sample_100.csv",
    index=False,
    encoding="utf-8-sig"
)

display(salary_check_sample.head(30))

,salary,salary_available,salary_min_million_vnd,salary_max_million_vnd,salary_avg_million_vnd,salary_currency,salary_parse_issue
165070,10.000.000 - 20.000.000 VND,1,10.0,20.0,15.0,VND,0
274446,15.000.000 - 35.000.000 VND,1,15.0,35.0,25.0,VND,0
116912,Đang cập nhật,0,NaN,NaN,NaN,Unknown,0
91059,18.000.000 - 25.000.000 VND,1,18.0,25.0,21.5,VND,0
188153,10.000.000 - 16.000.000 VND,1,10.0,16.0,13.0,VND,0
475508,12.000.000 - 18.000.000 VND,1,12.0,18.0,15.0,VND,0
47449,8.000.000 - 20.000.000 VND,1,8.0,20.0,14.0,VND,0
369572,3.000.000 - 5.000.000 VND,1,3.0,5.0,4.0,VND,0
433035,12.000.000 - 15.000.000 VND,1,12.0,15.0,13.5,VND,0
47028,7.000.000 - 10.000.000 VND,1,7.0,10.0,8.5,VND,0


In [76]:
text_check_sample = clean_train[
    [
        "job_title",
        "job_description",
        "requirements",
        "benefits"
    ]
].sample(100, random_state=42)

text_check_sample.to_csv(
    TABLE_DIR / "stage_03_text_clean_sample_100.csv",
    index=False,
    encoding="utf-8-sig"
)

display(text_check_sample.head(10))

,job_title,job_description,requirements,benefits
165070,giám sát thi công công trình nội thất,"khảo sát mặt bằng thi công, lập biện pháp thi ...","giới tính : nam, tuổi từ 25-35, tốt nghiệp:tru...",lương:thỏa thuận khi phỏng vấn ( dao động khoả...
274446,trợ lý giám đốc điều hành,hỗ trợ lãnh đạo trung quốc quản lý công việc h...,kỹ năng tiếng trung tốt (nghe-nói-đọc-viết); -...,lương tối thiểu 18 – 25tr tùy theo năng lực - ...
116912,dược sĩ cao đẳng (kho) - vinmec riverside,"hàng ngày theo dõi và ghi nhiệt độ, độ ẩm nơi ...",tốt nghiệp cao đẳng dược thành thạo tin học vă...,"laptop, chế độ bảo hiểm, du lịch, chế độ thưởn..."
91059,hành chính nhân sự,kinh doanh chuyên bán hàng ol và quản trị làm ...,nhanh nhẹn hoạt bát biết sử dụng máy tính,hưởng tháng lương thứ 13 du lịch 1 năm 1 lần 1...
188153,nhân viên kỹ thuật cơ khí tại củ chi - không y...,"tiếp nhận, nghiên cứu và làm rõ các thông tin ...","nam tốt nghiệp cao đẳng, đại học ngành kỹ thuậ...","mức lương: 10-16 triệu - thưởng lễ, thưởng tết..."
475508,nhân viên hành chính - tiếng trung (quận 7),1. tạo hạn mức trên hệ thống. 2. phân tích báo...,1. tốt nghiệp cao đẳng/ đại học trở lên chuyên...,cơm trưa: 50k/ngày laptop chế độ bảo hiểm phụ ...
47449,nhân viên tư vấn bán hàng - không yêu cầu kinh...,gọi điện tư vấn sản phẩm theo data khách hàng ...,có kinh nghiệm là một lợi thế (không có kinh n...,lương cứng lên đến 8tr + hoa hồng + thưởng nón...
369572,nhân viên phục vụ nhà hàng,"✅ part time: (7h-12h, 12h-18h, 18h-23h) - vị t...","nhanh nhẹn, có kinh nghiệm phục vụ là một ưu tiên",✅ quyền lợi: - sắp xếp ca linh hoạt hàng tuần ...
433035,nhân viên quay dựng / video editor,phụ trách quay phim và phối hợp với các bộ phậ...,kinh nghiệm: từ 1-3 năm trong lĩnh vực truyền ...,thu nhập: 12.000.000 – 15.000.000 vnđ + phụ cấ...
47028,nhân viên kế toán,"kiểm soát chi phí, áp dụng quy trình thanh toá...","trung cấp , cao đăng hoặc đại học các trường l...","thưởng lễ, tết, lương tháng 13 và các chế độ p..."


In [77]:
# ============================================================
# Export small audit files for external review
# ============================================================

AUDIT_DIR = OUTPUT_DIR / "audit_stage_03"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Sample clean data 1000 rows
clean_sample_1000 = clean_train.sample(
    n=min(1000, len(clean_train)),
    random_state=42
)

clean_sample_1000.to_csv(
    AUDIT_DIR / "stage_03_clean_data_sample_1000.csv",
    index=False,
    encoding="utf-8-sig"
)

# 2. Salary parse sample 200 rows
salary_parse_sample = clean_train[
    [
        "salary",
        "salary_available",
        "salary_min_million_vnd",
        "salary_max_million_vnd",
        "salary_avg_million_vnd",
        "salary_currency",
        "salary_parse_issue"
    ]
].sample(
    n=min(200, len(clean_train)),
    random_state=42
)

salary_parse_sample.to_csv(
    AUDIT_DIR / "stage_03_salary_parse_sample_200.csv",
    index=False,
    encoding="utf-8-sig"
)

# 3. Text clean sample 200 rows
text_clean_sample = clean_train[
    [
        "job_title",
        "job_description",
        "requirements",
        "benefits"
    ]
].sample(
    n=min(200, len(clean_train)),
    random_state=42
)

text_clean_sample.to_csv(
    AUDIT_DIR / "stage_03_text_clean_sample_200.csv",
    index=False,
    encoding="utf-8-sig"
)

# 4. Categorical value counts
categorical_cols = [
    "company_name",
    "location",
    "job_type",
    "job_industry",
    "experience_level",
    "education_level",
    "job_position",
    "salary_currency"
]

for col in categorical_cols:
    if col in clean_train.columns:
        vc = (
            clean_train[col]
            .value_counts(dropna=False)
            .head(100)
            .rename_axis(col)
            .reset_index(name="count")
        )

        vc["ratio"] = vc["count"] / len(clean_train)

        vc.to_csv(
            AUDIT_DIR / f"stage_03_value_counts_{col}.csv",
            index=False,
            encoding="utf-8-sig"
        )

# 5. Numeric describe
numeric_cols = [
    "salary_min_million_vnd",
    "salary_max_million_vnd",
    "salary_avg_million_vnd",
    "salary_parse_issue",
    "job_title_char_len",
    "job_title_word_count",
    "job_description_char_len",
    "job_description_word_count",
    "requirements_char_len",
    "requirements_word_count",
    "benefits_char_len",
    "benefits_word_count",
    "year"
]

existing_numeric_cols = [col for col in numeric_cols if col in clean_train.columns]

numeric_describe = (
    clean_train[existing_numeric_cols]
    .describe()
    .T
    .reset_index()
    .rename(columns={"index": "column"})
)

numeric_describe.to_csv(
    AUDIT_DIR / "stage_03_numeric_describe.csv",
    index=False,
    encoding="utf-8-sig"
)

# 6. Final schema
schema_df = pd.DataFrame({
    "column": clean_train.columns,
    "dtype": [str(dtype) for dtype in clean_train.dtypes],
    "missing_count": clean_train.isna().sum().values,
    "missing_ratio": clean_train.isna().mean().values,
})

schema_df.to_csv(
    AUDIT_DIR / "stage_03_clean_schema.csv",
    index=False,
    encoding="utf-8-sig"
)

numeric_cols = [
    "salary_min_million_vnd",
    "salary_max_million_vnd",
    "salary_avg_million_vnd",
    "salary_parse_issue",
    "job_title_char_len",
    "job_title_word_count",
    "job_description_char_len",
    "job_description_word_count",
    "requirements_char_len",
    "requirements_word_count",
    "benefits_char_len",
    "benefits_word_count",
    "year"
]

print("Saved audit files to:", AUDIT_DIR)

Saved audit files to: D:\DataScientFinalProject\outputs\audit_stage_03
